# Tratamento de Registros de Consumo

Este notebook processa os arquivos CSV da pasta `data/trat_consumo/` e os consolida em um único arquivo no mesmo padrão dos arquivos `.xlsx` da pasta `data/consumos/`.

**Colunas do padrão de saída:**
- `MATRÍCULA IQ` — matrícula do aluno
- `DATA` — data do consumo (datetime)
- `NOME DO ALUNO` — nome completo
- `TURMA` — turma do aluno

---
**Instruções:**
1. Rode a célula de teste (Seção 2) para validar o processamento de **um único arquivo**.
2. Após validar, rode a Seção 3 para processar **todos os arquivos** e gerar o consolidado.

## 1. Configuração e Funções de Tratamento

In [ ]:
import pandas as pd
import numpy as np
import os
import re
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# ─── Caminhos (USANDO BARRAS NORMAIS PARA EVITAR BUG NO JSON) ──────────────
BASE_DIR      = Path('D:/PROJETOS/PREDICAO/novos_experimentos')
INPUT_DIR     = BASE_DIR / 'data' / 'trat_consumo'
OUTPUT_DIR    = BASE_DIR / 'data' / 'consumos'
OUTPUT_FILE   = OUTPUT_DIR / 'consumos_trat.xlsx'
REF_FILE_2023 = OUTPUT_DIR / 'consumos_2023.xlsx'

print('Pasta de entrada:', INPUT_DIR)
print('Arquivo de saída: ', OUTPUT_FILE)
print(f'Arquivos CSV encontrados: {len(list(INPUT_DIR.glob("*.csv")))} arquivos')

In [ ]:
# ─── Colunas de saída padrão ─────────────────────────────────────────────────
COLUNAS_SAIDA = ['MATRÍCULA IQ', 'DATA', 'NOME DO ALUNO', 'TURMA']

def normalizar_matricula(valor):
    if pd.isna(valor):
        return np.nan
    s = str(valor).strip().upper()
    if re.match(r'^\d{7,}', s):
        s = 'IQ' + s
    return s if s.startswith('IQ') else np.nan

def detectar_formato(df):
    if df is None or df.empty or df.shape[1] < 2:
        return 'VAZIO'
    ncols = df.shape[1]
    primeira = df.iloc[0]
    if ncols >= 47:
        return 'FORM'
    col0 = str(primeira.iloc[0]).strip()
    if ncols >= 5:
        if col0.upper().startswith('IQ'):
            return 'C11' if ncols > 5 else 'A5'
        col3 = str(primeira.iloc[3]).strip() if ncols > 3 else ''
        if col3.upper().startswith('IQ'):
            return 'B5'
        if re.match(r'^\d{5,}$', col0):
            return 'B5'
    return 'DESCONHECIDO'

def parsear_data(valor):
    if pd.isna(valor):
        return pd.NaT
    s = str(valor).strip()[:10]
    for fmt in ('%d/%m/%Y', '%Y/%m/%d', '%Y-%m-%d', '%d-%m-%Y'):
        try:
            return pd.to_datetime(s, format=fmt)
        except:
            pass
    return pd.NaT

def processar_formato_A5(df):
    resultado = pd.DataFrame()
    resultado['MATRÍCULA IQ'] = df.iloc[:, 0].apply(normalizar_matricula)
    resultado['DATA']         = df.iloc[:, 1].apply(parsear_data)
    resultado['NOME DO ALUNO']= df.iloc[:, 3].astype(str).str.strip()
    resultado['TURMA']        = df.iloc[:, 4].astype(str).str.strip()
    return resultado

def processar_formato_B5(df):
    resultado = pd.DataFrame()
    resultado['MATRÍCULA IQ'] = df.iloc[:, 3].apply(normalizar_matricula)
    resultado['DATA']         = df.iloc[:, 4].apply(parsear_data)
    resultado['NOME DO ALUNO']= df.iloc[:, 1].astype(str).str.strip()
    resultado['TURMA']        = df.iloc[:, 2].astype(str).str.strip()
    return resultado

def processar_formato_C11(df):
    resultado = pd.DataFrame()
    resultado['MATRÍCULA IQ'] = df.iloc[:, 0].apply(normalizar_matricula)
    resultado['DATA']         = df.iloc[:, 1].apply(parsear_data)
    resultado['NOME DO ALUNO']= df.iloc[:, 3].astype(str).str.strip()
    resultado['TURMA']        = df.iloc[:, 4].astype(str).str.strip()
    return resultado

def extrair_data_do_cabecalho(df):
    datas_colunas = []
    for i, col in enumerate(df.columns):
        col_str = str(col).strip()
        match = re.search(r'(\d{2}/\d{2}/\d{4}|\d{4}/\d{2}/\d{2})', col_str)
        if match:
            dt = parsear_data(match.group(1))
            if dt is not pd.NaT:
                datas_colunas.append((i, dt))
    return datas_colunas

def processar_formato_FORM(df_raw):
    primeiro_val = str(df_raw.iloc[0, 0]).strip()
    tem_header = re.match(r'\d{4}/\d{2}/\d{2}', primeiro_val) is not None
    if tem_header:
        df = df_raw.copy()
        col_matricula, col_nome, col_turma = 4, 5, 6
    else:
        df = df_raw.copy()
        df.columns = df.iloc[0]
        df = df.iloc[1:].reset_index(drop=True)
        col_matricula, col_nome, col_turma = 4, 5, 6

    datas_colunas = extrair_data_do_cabecalho(df_raw)
    linhas = []
    for _, row in df.iterrows():
        matricula = normalizar_matricula(row.iloc[col_matricula])
        nome = str(row.iloc[col_nome]).strip()
        turma = str(row.iloc[col_turma]).strip()
        if datas_colunas:
            for col_idx, dt in datas_colunas:
                val = row.iloc[col_idx]
                if not pd.isna(val) and str(val).strip() not in ('', 'nan'):
                    linhas.append({'MATRÍCULA IQ': matricula, 'DATA': dt, 'NOME DO ALUNO': nome, 'TURMA': turma})
        else:
            datas_texto = str(row.iloc[3])
            datas_encontradas = re.findall(r'\d{2}/\d{2}/\d{4}', datas_texto)
            for d in datas_encontradas:
                dt = parsear_data(d)
                linhas.append({'MATRÍCULA IQ': matricula, 'DATA': dt, 'NOME DO ALUNO': nome, 'TURMA': turma})
    if not linhas:
        return pd.DataFrame(columns=COLUNAS_SAIDA)
    return pd.DataFrame(linhas)[COLUNAS_SAIDA]

def processar_csv(caminho):
    arquivo = Path(caminho)
    if arquivo.stat().st_size < 100:
        print(f'  [IGNORADO] {arquivo.name} — arquivo muito pequeno ({arquivo.stat().st_size} bytes)')
        return None
    
    try:
        df_raw = pd.read_csv(caminho, header=None, encoding='utf-8')
    except UnicodeDecodeError:
        df_raw = pd.read_csv(caminho, header=None, encoding='latin1')
    except Exception as e:
        print(f'  [ERRO LEITURA] {arquivo.name}: {e}')
        return None

    if df_raw.empty:
        print(f'  [VAZIO] {arquivo.name}')
        return None

    formato = detectar_formato(df_raw)
    try:
        if formato == 'A5': df_saida = processar_formato_A5(df_raw)
        elif formato == 'B5': df_saida = processar_formato_B5(df_raw)
        elif formato == 'C11': df_saida = processar_formato_C11(df_raw)
        elif formato == 'FORM': df_saida = processar_formato_FORM(df_raw)
        else:
            print(f'  [DESCONHECIDO] {arquivo.name} — {df_raw.shape[1]} colunas. Pulando.')
            return None
    except Exception as e:
        print(f'  [ERRO PROCESSAMENTO] {arquivo.name} (formato={formato}): {e}')
        return None

    df_saida = df_saida.dropna(subset=['MATRÍCULA IQ', 'DATA'])
    df_saida = df_saida[df_saida['MATRÍCULA IQ'] != 'nan']
    print(f'  [OK] {arquivo.name:<15} | formato={formato:<4} | {len(df_saida)} registros')
    return df_saida

print('✅ Funções carregadas com sucesso!')

## 2. TESTE — Processar apenas um arquivo para validar

> ⚠️ **Valide este resultado antes de executar a Seção 3 (todos os arquivos).**

In [ ]:
# ─── PILOTO: escolha qualquer arquivo para testar ─────────────────────────────
ARQUIVO_PILOTO = INPUT_DIR / '100.csv'

print(f'Processando arquivo piloto: {ARQUIVO_PILOTO.name}')
print('-' * 60)
df_piloto = processar_csv(ARQUIVO_PILOTO)

if df_piloto is not None and not df_piloto.empty:
    print(f'\n📋 Resultado ({len(df_piloto)} registros):')
    display(df_piloto.head(10))
    print(f'\nTipos de coluna:')
    print(df_piloto.dtypes)
    print(f'\nPrimeira data: {df_piloto["DATA"].min()}')
    print(f'Última data:   {df_piloto["DATA"].max()}')
else:
    print('⚠️ Nenhum registro gerado para este arquivo.')

In [ ]:
print('=== PADRÃO DE REFERÊNCIA (consumos_2023.xlsx) — primeiras 5 linhas ===')
df_ref = pd.read_excel(REF_FILE_2023)
display(df_ref.head(5))
print(f'\nDtypes de referência:')
print(df_ref.dtypes)

## 3. Processar TODOS os arquivos e consolidar

> ⚠️ **Execute somente após validar a Seção 2.**

In [ ]:
arquivos_csv = sorted(INPUT_DIR.glob('*.csv'))
print(f'Total de arquivos CSV: {len(arquivos_csv)}')
print('=' * 60)

lista_dfs = []
nao_processados = []

for csv_path in arquivos_csv:
    df_proc = processar_csv(csv_path)
    if df_proc is not None and not df_proc.empty:
        df_proc['arquivo_origem'] = csv_path.name
        lista_dfs.append(df_proc)
    else:
        nao_processados.append(csv_path.name)

print('=' * 60)
print(f'\nArquivos processados com sucesso: {len(lista_dfs)}')
if nao_processados:
    print('  Não processados:', nao_processados)

In [ ]:
if not lista_dfs:
    print('⚠️ Nenhum arquivo foi processado.')
else:
    df_consolidado = pd.concat(lista_dfs, ignore_index=True)
    print(f'Total de registros (antes de deduplicar): {len(df_consolidado)}')

    df_final = df_consolidado[COLUNAS_SAIDA].copy()
    df_final = df_final.drop_duplicates(subset=['MATRÍCULA IQ', 'DATA'])
    df_final['DATA'] = pd.to_datetime(df_final['DATA'], errors='coerce')
    df_final = df_final.sort_values(['DATA', 'MATRÍCULA IQ']).reset_index(drop=True)

    # --- Padronização das Turmas ---
    df_final['TURMA'] = df_final['TURMA'].str.strip()
    mapa_turmas = {
        '1.º A': '1º A - MEC',
        '1.º B': '1º B - MEC',
        '2.º A': '2º A - MEC',
        '2.º B': '2º B - MEC',
        '3.º A': '3º A - MEC',
        '3.º B': '3º B - MEC',
        '1º A': '1º A - MEC',
        '1º B': '1º B - MEC',
        '2º A': '2º A - MEC',
        '2º B': '2º B - MEC',
        '3º A': '3º A - MEC',
        '3º B': '3º B - MEC',
        '1A': '1º A - MEC',
        '1B': '1º B - MEC',
        '2A': '2º A - MEC',
        '2B': '2º B - MEC',
        '3A': '3º A - MEC',
        '3B': '3º B - MEC'
    }
    df_final['TURMA'] = df_final['TURMA'].replace(mapa_turmas)
    df_final['TURMA'] = df_final['TURMA'].str.replace(r' (Técnico|TÃ©cnico).*', ' - MEC', regex=True)

    print(f'\nDistribuição por ano antes de conversão para string:')
    print(df_final['DATA'].dt.year.value_counts().sort_index())

    # Formata a coluna DATA para string DD/MM/AAAA
    df_final['DATA'] = df_final['DATA'].dt.strftime('%d/%m/%Y')

    print(f'Total de registros (após deduplicar):    {len(df_final)}')
    print(f'\nDistribuição de Turmas Padrão:')
    print(df_final['TURMA'].value_counts())
    print(f'\nPrimeiras linhas:')
    display(df_final.head(10))

In [ ]:
# ─── Salvar para Excel ────────────────────────────────────────────────────────
df_final.to_excel(OUTPUT_FILE, index=False)
print(f'✅ Arquivo salvo em: {OUTPUT_FILE}')
print(f'   Total de registros: {len(df_final)}')

print(f'\n=== Verificação de compatibilidade ===')
df_ref = pd.read_excel(REF_FILE_2023)
print(f'Referência (2023):  colunas={list(df_ref.columns)}')
print(f'Arquivo gerado:     colunas={list(df_final.columns)}')
print(f'\nCompatível: {list(df_ref.columns) == list(df_final.columns)}')